In [5]:
import pandas as pd
import numpy as np
import wandb

# 1. Log in to W&B (This will prompt for your API key in Colab)
wandb.login()

# 2. Initialize the W&B run
# Replace 'your-project-name' with what you want to call this project in W&B
run = wandb.init(
    entity="mmcgee18-georgia-state-university",
    project="monte-carlo-health-impact",
    name="baseline-simulation-1000",
    save_code=True, # Tells W&B to try and capture the notebook code
    config={
        "n_iterations": 1000,
        "adoption_min": 0.01,
        "adoption_mode": 0.02,
        "adoption_max": 0.03,
        "features": ['diabetes_pct', 'high_bp_pct', 'high_cholesterol_pct', 'asthma_pct', 'no_checkup_pct']
    }
)

csv_url = 'https://raw.githubusercontent.com/ctrazona1385/DS_Capstone_Group_1/main/data/v2_cleaned/all_merged.csv'
df = pd.read_csv(csv_url)

display(df.head())

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aidanrmoore2 (mmcgee18-georgia-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


,index,geoid_tract,uninsured_pct,high_bp_pct,asthma_pct,no_checkup_pct,no_dental_visit_pct,diabetes_pct,high_cholesterol_pct,geoid,...,median_household_income,hpsa_score_max,hpsa_designated,poverty_rate_pct,total_population_poverty,md_primary_care_count,do_primary_care_count,total_population,pcp_count,pcp_per_100k
0,226,1001020100,15.0,38.8,9.8,74.1,63.4,10.7,35.6,1001,...,72481.0,15.0,1.0,11.8,6988.0,24.0,1.0,59759.0,25.0,41.834703
1,2109,1001020200,19.9,43.6,10.9,76.5,55.6,13.4,32.7,1001,...,72481.0,15.0,1.0,11.8,6988.0,24.0,1.0,59759.0,25.0,41.834703
2,3955,1001020300,16.7,39.6,10.1,74.3,61.1,11.3,35.0,1001,...,72481.0,15.0,1.0,11.8,6988.0,24.0,1.0,59759.0,25.0,41.834703
3,5732,1001020400,12.5,39.2,8.8,75.5,70.3,10.2,37.7,1001,...,72481.0,15.0,1.0,11.8,6988.0,24.0,1.0,59759.0,25.0,41.834703
4,8015,1001020500,12.8,34.6,9.2,73.5,68.4,8.7,32.7,1001,...,72481.0,15.0,1.0,11.8,6988.0,24.0,1.0,59759.0,25.0,41.834703


In [6]:


# Define variables in a strict order to maintain matrix alignment
features = [
    'diabetes_pct',
    'high_bp_pct',
    'high_cholesterol_pct',
    'asthma_pct',
    'no_checkup_pct'
]

# Ensure we only process rows that have no missing values for these features
df = df.dropna(subset=features).copy()
n_tracts = len(df)
n_iterations = 1000 # Reduced for memory management; increase if your RAM allows

# 2. Define Parameters (Means and Standard Deviations)
means = np.array([0.15, 0.12, 0.17, 0.12, 0.25])
stdevs = np.array([0.04, 0.03, 0.04, 0.03, 0.05])
adoption_params = {'min': 0.01, 'mode': 0.02, 'max': 0.03}

# 3. Build the Correlation and Covariance Matrices
# Extracted exactly from the provided image for the 5 selected features
correlation_matrix = np.array([
    # Diab  HBP   Chol  Asth  Checkup
    [1.00, 0.87, 0.52, 0.55, 0.42],  # Diabetes
    [0.87, 1.00, 0.74, 0.45, 0.64],  # High BP
    [0.52, 0.74, 1.00, -0.03, 0.49], # High Chol
    [0.55, 0.45, -0.03, 1.00, 0.24], # Asthma
    [0.42, 0.64, 0.49, 0.24, 1.00]   # No Checkup
])

# Covariance = Correlation * Outer Product of Standard Deviations
covariance_matrix = correlation_matrix * np.outer(stdevs, stdevs)

# Extract baselines into a numpy array (Shape: n_tracts x 5)
baseline_matrix = df[features].values

# Prepare a 3D array to store the results: (n_tracts, n_features, n_iterations)
simulation_results = np.zeros((n_tracts, len(features), n_iterations))

# 4. Run the Monte Carlo Simulation
print(f"Running {n_iterations} iterations for {n_tracts} census tracts...")

for i in range(n_iterations):
    # A. Draw tract-level adoption rates (Shape: n_tracts x 1)
    adoption_rates = np.random.triangular(
        adoption_params['min'],
        adoption_params['mode'],
        adoption_params['max'],
        size=(n_tracts, 1)
    )

    # B. Draw correlated tract-level reduction efficiencies (Shape: n_tracts x 5)
    # This assumes the intervention's success varies slightly per tract but remains
    # highly correlated across diseases within that specific tract.
    reductions = np.random.multivariate_normal(means, covariance_matrix, size=n_tracts)

    # Floor reductions at 0 (an intervention shouldn't increase disease prevalence)
    reductions = np.maximum(0, reductions)

    # C. Calculate Effective Improvement (Shape: n_tracts x 5)
    effective_improvement = adoption_rates * reductions

    # D. Apply to baselines to get Absolute Percentage Point Reductions
    simulation_results[:, :, i] = baseline_matrix * effective_improvement

# 5. Aggregate Results
results_df = df[['FIPS']].copy() if 'FIPS' in df.columns else pd.DataFrame(index=df.index)

for idx, feature in enumerate(features):
    # Extract the simulated reductions for this specific feature across all iterations
    feature_sims = simulation_results[:, idx, :]

    # Calculate Mean, 5th, and 95th percentiles (absolute percentage point drops)
    results_df[f'{feature}_reduction_mean'] = np.mean(feature_sims, axis=1)
    results_df[f'{feature}_reduction_p05'] = np.percentile(feature_sims, 5, axis=1)
    results_df[f'{feature}_reduction_p95'] = np.percentile(feature_sims, 95, axis=1)

print("Simulation Complete. Sample output:")
print(results_df.head())

# --- ADD THIS TO THE BOTTOM OF YOUR SIMULATION CELL ---

print("Uploading results to Weights & Biases...")

# 1. Log high-level summary metrics (Average reductions across all 71k tracts)
# This will create nice line charts/bar charts in your W&B dashboard
wandb.log({
    "avg_diabetes_reduction": results_df['diabetes_pct_reduction_mean'].mean(),
    "avg_high_bp_reduction": results_df['high_bp_pct_reduction_mean'].mean(),
    "avg_high_cholesterol_reduction": results_df['high_cholesterol_pct_reduction_mean'].mean(),
    "avg_asthma_reduction": results_df['asthma_pct_reduction_mean'].mean(),
    "avg_no_checkup_reduction": results_df['no_checkup_pct_reduction_mean'].mean()
})

# 2. Save a sample of the results as an interactive W&B Table
# We take the first 100 rows to prevent the table from being too massive
results_table = wandb.Table(dataframe=results_df.head(100))
wandb.log({"sample_simulated_tracts": results_table})

# 3. Explicitly save the notebook file itself to the W&B run files
# Make sure the string exactly matches your file name in Colab
wandb.save("Monte_Carlo_Health_Impact.ipynb")

# 4. Finish the run to sync everything
wandb.finish()

print("W&B Sync Complete!")

Running 1000 iterations for 71506 census tracts...
Simulation Complete. Sample output:
   diabetes_pct_reduction_mean  diabetes_pct_reduction_p05  \
0                     0.031932                    0.015918   
1                     0.039709                    0.020033   
2                     0.033775                    0.016649   
3                     0.030307                    0.015103   
4                     0.026438                    0.012818   

   diabetes_pct_reduction_p95  high_bp_pct_reduction_mean  \
0                    0.052009                    0.092192   
1                    0.064365                    0.103811   
2                    0.053343                    0.094674   
3                    0.048506                    0.092473   
4                    0.042130                    0.083757   

   high_bp_pct_reduction_p05  high_bp_pct_reduction_p95  \
0                   0.049232                   0.144350   
1                   0.052801                   0.165637

avg_asthma_reduction,▁
avg_diabetes_reduction,▁
avg_high_bp_reduction,▁
avg_high_cholesterol_reduction,▁
avg_no_checkup_reduction,▁
avg_asthma_reduction,0.02406
avg_diabetes_reduction,0.03248
avg_high_bp_reduction,0.07796
avg_high_cholesterol_reduction,0.10817
avg_no_checkup_reduction,0.36897


W&B Sync Complete!
